# 04 · Chapter 3: Meet the junior assistant, a free LLM on my laptop (Ollama)

Between "tiny ML model" and "big cloud LLM" there's a middle option: a **small LLM running on my laptop**.
- Free (no per-token cost)
- Private (data never leaves my computer)
- Works offline
- But less smart than the big cloud models

**Setup (once, in the terminal):**
```
1. Install Ollama from https://ollama.com/download (Mac, Windows or Linux)
2. Mac/Windows: open the Ollama app. Linux: run  ollama serve
3. ollama pull llama3.2:3b     (downloads the model once, about 2 GB)

Full steps for every system: docs/install-guide.md
```

In [1]:
import sys
sys.path.append("..")   # so Python can find the budget_agent folder

import pandas as pd
pd.set_option("display.max_colwidth", 80)
import requests
from budget_agent import config

OLLAMA = "http://localhost:11434"
MODEL = config.LOCAL_MODEL   # "llama3.2:3b", the same model the app uses
requests.get(f"{OLLAMA}/api/version").json()

{'version': '0.35.1'}

## Step 1: Which models do I have?

In [2]:
for m in requests.get(f"{OLLAMA}/api/tags").json()["models"]:
    print(m["name"], f"{m['size'] / 1e9:.1f} GB")

llama3.2:3b 2.0 GB


## Step 2: Why is it only 2 GB? (quantization)

The model has about 3 billion numbers ("parameters").
Stored normally (16 bits each) that's about 6 GB. Ollama stores each number in about **4 bits** instead → about 2 GB.
Slightly less accurate, but it fits in a laptop's memory and runs much faster. This trick is called **quantization**.

In [3]:
info = requests.post(f"{OLLAMA}/api/show", json={"model": MODEL}).json()
info["details"]

{'parent_model': '',
 'format': 'gguf',
 'family': 'llama',
 'families': ['llama'],
 'parameter_size': '3.2B',
 'quantization_level': 'Q4_K_M'}

## Step 3: Ask it something

In [4]:
import time

def ask_local(question, system="You are a helpful assistant. Answer in 1-2 sentences."):
    start = time.perf_counter()
    reply = requests.post(f"{OLLAMA}/api/chat", json={
        "model": MODEL,
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": question}],
        "stream": False,
    }).json()
    seconds = time.perf_counter() - start
    words_per_sec = reply["eval_count"] / (reply["eval_duration"] / 1e9)
    print(reply["message"]["content"])
    print(f"\n⏱ {seconds:.2f}s   ⚡ {words_per_sec:.0f} tokens/second   💲 $0")

ask_local("What is an AI agent?")

An AI agent is a computer program designed to perform tasks based on its programming, environment, and interactions with users or other agents. It is a type of artificial intelligence that can make decisions, learn from data, and adapt to changing situations, often with goals and objectives to achieve.

⏱ 3.01s   ⚡ 53 tokens/second   💲 $0


The first call is slower because the model is loading into memory. Let me run it again:

(`LocalLLM.chat()` in `budget_agent/llm.py` sends this same request; the router uses it for the junior assistant.)

In [5]:
ask_local("What is an AI agent?")

An AI agent is a computer program that can perceive its environment, make decisions, and act upon them, often in a self-directed and autonomous manner, without being explicitly programmed for every possible scenario. AI agents can be designed to learn from experience and adapt to new situations, making them increasingly sophisticated and interactive.

⏱ 1.20s   ⚡ 53 tokens/second   💲 $0


## Step 4: Can it answer NovaMart questions?

I give it our policy and ask simple questions.

In [6]:
from budget_agent.data import load_policy

shop_system = "You are NovaMart's support assistant. Answer in 1-3 short sentences using ONLY this policy:\n\n" + load_policy()

ask_local("can i pay with cash on delivery?", shop_system)

Yes, you can pay with Cash on Delivery (COD) on orders up to ₹5,000.

⏱ 1.15s   ⚡ 55 tokens/second   💲 $0


In [7]:
ask_local(config.DEMO_MESSAGES["local_llm"], shop_system)   # kitne din me delivery hoti hai

Standard delivery me 3-5 kaalik din hoti hai, jismein ₹499 se adhik order me free hoti hai.

⏱ 0.66s   ⚡ 55 tokens/second   💲 $0


In [8]:
ask_local("how many days do i have to return something", shop_system)

You have **30 days** from the date of delivery to return an item.

⏱ 0.38s   ⚡ 56 tokens/second   💲 $0


## Step 5: What it should NOT do

A small model can sound confident and still be wrong. And it has no tools: it can't look up orders or start refunds.
So in the support office, the junior only answers **policy questions**. Order questions and anything that changes an order go to the senior expert.

In [9]:
ask_local("refund my order NM10002 now", shop_system)   # it can't really do this!

To process your refund, please provide your order number, NM10002, and we will look into it. The refund will be processed in 5-7 working days, and it will go back to your original payment method.

⏱ 0.95s   ⚡ 53 tokens/second   💲 $0


## What I learned
- A 3B model runs on a laptop thanks to **quantization**, for $0.
- Good for simple policy questions, even Hinglish.
- Not trusted for actions or order questions. That's the senior expert's job.

Next: put the whole office together → `05_the_router.ipynb`